In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/ecommerce-analytics'

df = pd.read_csv(f'{BASE}/data/processed/stage1_loaded.csv')

print(f"Loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")

Mounted at /content/drive
Loaded: 1,067,371 rows
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'TotalRevenue']


In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],
                                   errors='coerce')

# Extract useful time columns for later analysis
df['Year']  = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day']   = df['InvoiceDate'].dt.day_name()
df['Hour']  = df['InvoiceDate'].dt.hour

print("InvoiceDate dtype:", df['InvoiceDate'].dtype)
print("Date range:", df['InvoiceDate'].min(),
      "→", df['InvoiceDate'].max())
print("Sample:")
print(df[['InvoiceDate','Year','Month','Day','Hour']].head(3))

InvoiceDate dtype: datetime64[ns]
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Sample:
          InvoiceDate  Year  Month      Day  Hour
0 2009-12-01 07:45:00  2009     12  Tuesday     7
1 2009-12-01 07:45:00  2009     12  Tuesday     7
2 2009-12-01 07:45:00  2009     12  Tuesday     7


In [ ]:
before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)
removed = before - after

print(f"Rows before  : {before:,}")
print(f"Rows after   : {after:,}")
print(f"Duplicates removed: {removed:,}")

Rows before  : 1,067,371
Rows after   : 1,033,036
Duplicates removed: 34,335


In [ ]:
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
}).query('missing_count > 0')
print(summary)

# What does a row with no Customer ID look like?
print("\nSample rows with missing Customer ID:")
print(df[df['Customer ID'].isnull()][
    ['Invoice','Description','Quantity','Price']
].head(5))

=== MISSING VALUES ===
             missing_count  missing_%
Description           4275       0.41
Customer ID         235151      22.76

Sample rows with missing Customer ID:
    Invoice                Description  Quantity  Price
263  489464               85123a mixed       -96   0.00
283  489463                      short      -240   0.00
284  489467                21733 mixed      -192   0.00
470  489521                        NaN       -50   0.00
577  489525  BLUE PULL BACK RACING CAR         1   0.55


In [ ]:
# ALL transactions — for product & revenue analysis
df_all = df.copy()

# Only identified customers — for RFM & churn analysis
df_customers = df[df['Customer ID'].notnull()].copy()

# Fill missing Description with 'Unknown'
df_all['Description'].fillna('Unknown', inplace=True)
df_customers['Description'].fillna('Unknown', inplace=True)

print(f"df_all       : {len(df_all):,} rows (all transactions)")
print(f"df_customers : {len(df_customers):,} rows (identified customers only)")

/tmp/ipykernel_155/2616286335.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_all['Description'].fillna('Unknown', inplace=True)
/tmp/ipykernel_155/2616286335.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

df_all       : 1,033,036 rows (all transactions)
df_customers : 797,885 rows (identified customers only)


In [ ]:
# Cancellations have Invoice starting with 'C'
df_all['IsCancelled'] = df_all['Invoice'].astype(str).str.startswith('C')

# Separate
df_returns  = df_all[df_all['IsCancelled'] == True].copy()
df_sales    = df_all[df_all['IsCancelled'] == False].copy()

# Also filter out negative quantities in sales
df_sales = df_sales[df_sales['Quantity'] > 0]
df_sales = df_sales[df_sales['Price'] > 0]

print(f"Sales rows    : {len(df_sales):,}")
print(f"Return rows   : {len(df_returns):,}")
print(f"Return rate   : {len(df_returns)/len(df_all)*100:.1f}%")

Sales rows    : 1,007,913
Return rows   : 19,104
Return rate   : 1.8%


In [ ]:
print("=== QUANTITY STATS ===")
print(df_sales['Quantity'].describe())

print("\n=== PRICE STATS ===")
print(df_sales['Price'].describe())

# Flag extreme outliers (beyond 99.9th percentile)
q_upper = df_sales['Quantity'].quantile(0.999)
p_upper = df_sales['Price'].quantile(0.999)

print(f"\nQuantity 99.9th percentile : {q_upper:.0f}")
print(f"Price    99.9th percentile : £{p_upper:.2f}")

outliers = df_sales[
    (df_sales['Quantity'] > q_upper) |
    (df_sales['Price'] > p_upper)
]
print(f"\nOutlier rows flagged: {len(outliers):,}")

=== QUANTITY STATS ===
count    1.007913e+06
mean     1.111718e+01
std      1.284700e+02
min      1.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.200000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

=== PRICE STATS ===
count    1.007913e+06
mean     4.074252e+00
std      5.043045e+01
min      1.000000e-03
25%      1.250000e+00
50%      2.100000e+00
75%      4.130000e+00
max      2.511109e+04
Name: Price, dtype: float64

Quantity 99.9th percentile : 500
Price    99.9th percentile : £160.03

Outlier rows flagged: 1,960


In [ ]:
df_sales['Quantity'] = df_sales['Quantity'].clip(
    upper=df_sales['Quantity'].quantile(0.999)
)
df_sales['Price'] = df_sales['Price'].clip(
    upper=df_sales['Price'].quantile(0.999)
)

# Recalculate TotalRevenue after capping
df_sales['TotalRevenue'] = df_sales['Quantity'] * df_sales['Price']

print("Outliers capped.")
print(f"Max Quantity now : {df_sales['Quantity'].max():.0f}")
print(f"Max Price now    : £{df_sales['Price'].max():.2f}")

Outliers capped.
Max Quantity now : 500
Max Price now    : £160.03


In [ ]:
print("=" * 45)
print("  STAGE 2 CLEANING SUMMARY")
print("=" * 45)
print(f"  Raw rows loaded       : 1,067,371")
print(f"  After deduplication   : {len(df_all):,}")
print(f"  Clean sales rows      : {len(df_sales):,}")
print(f"  Return rows           : {len(df_returns):,}")
print(f"  Identified customers  : {len(df_customers):,}")
print(f"  Countries             : {df_sales['Country'].nunique()}")
print(f"  Date range            : {df_sales['InvoiceDate'].min().date()} → {df_sales['InvoiceDate'].max().date()}")
print(f"  Total clean revenue   : £{df_sales['TotalRevenue'].sum():,.0f}")
print("=" * 45)
print("  Missing values remaining:")
print(df_sales.isnull().sum()[df_sales.isnull().sum() > 0])

  STAGE 2 CLEANING SUMMARY
  Raw rows loaded       : 1,067,371
  After deduplication   : 1,033,036
  Clean sales rows      : 1,007,913
  Return rows           : 19,104
  Identified customers  : 797,885
  Countries             : 43
  Date range            : 2009-12-01 → 2011-12-09
  Total clean revenue   : £19,340,686
  Missing values remaining:
Customer ID    228488
dtype: int64


In [ ]:
df_sales.to_csv(
    f'{BASE}/data/processed/stage2_sales_clean.csv',
    index=False)

df_returns.to_csv(
    f'{BASE}/data/processed/stage2_returns.csv',
    index=False)

df_customers.to_csv(
    f'{BASE}/data/processed/stage2_customers.csv',
    index=False)

print("Stage 2 DONE! Files saved to Drive:")
print("  stage2_sales_clean.csv  ← use this for all analysis")
print("  stage2_returns.csv      ← return rate analysis")
print("  stage2_customers.csv    ← RFM segmentation")

Stage 2 DONE! Files saved to Drive:
  stage2_sales_clean.csv  ← use this for all analysis
  stage2_returns.csv      ← return rate analysis
  stage2_customers.csv    ← RFM segmentation
